# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/codealchemist007/week1_assignment1/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane (Lane 2: Refresh/Content Opportunity Scoring) maps to two task types
combined: classification and ranking. At the core, it's a classification
problem - predicting whether a page will decline, based on observed signals.
But the actual output isn't just a yes/no label - it's used to rank pages
so a reviewer knows which ones to look at first. So the task type is really
"classification whose output feeds a ranking/scoring step", which matches
how the starter pipeline already works (train a classifier, use its
probability as the score to rank pages).

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The target I'm working toward is a proxy for now: is_declining_label, which
comes from trend_direction == "down" in the starter dataset. Per the data
skill, trend_direction (and the trend_pct it's derived from) can only ever
be a target, never a feature - so this rule keeps me from accidentally
leaking the answer into my own inputs.

That said, this is explicitly a weak proxy since it just looks at the
current window, not a real future outcome. The stronger version I want to
build toward eventually is a genuinely observed future label - like
"will this page's impressions/clicks decline over the next 30 days,"
built from the daily warehouse data. For this notebook though, I'm using
the starter's existing proxy label since that's what's available in the
30k-row starter dataset.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

The metric that matches this decision is precision@K - specifically
precision@50, matching what the starter pipeline already reports. This
fits because a reviewer only has time to check a limited number of pages
(say the top 50), so what actually matters is: out of the pages ranked at
the very top, how many are truly worth reviewing? That's a better fit than
a generic accuracy score, which doesn't care about ranking order or
capacity limits. The starter's own numbers show this works - random forest
hit 0.740 precision@50 vs 0.240 for the plain rule baseline.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
!git clone https://github.com/codealchemist007/flyrank-ml-track.git
df = pd.read_csv("flyrank-ml-track/data/raw/content_refresh_anonymized.csv")

# Show the unit of analysis - one row = one page
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
display(df.head())

# What the target/proxy column looks like on its own
df["is_declining_label"] = df["trend_direction"] == "down"
df[["content_id", "trend_direction", "is_declining_label"]].head(10)

Cloning into 'flyrank-ml-track'...
remote: Enumerating objects: 137, done.
remote: Counting objects: 100% (137/137), done.
remote: Compressing objects: 100% (93/93), done.
remote: Total 137 (delta 49), reused 92 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (137/137), 1.85 MiB | 7.36 MiB/s, done.
Resolving deltas: 100% (49/49), done.
Shape: 30000 rows, 44 columns


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


,content_id,trend_direction,is_declining_label
0,content_304f48230142,down,True
1,content_a1fb4e703a9e,down,True
2,content_9aa793d4d895,down,True
3,content_331d6c4de07b,stable,False
4,content_d99b7a2d90ca,down,True
5,content_d4084a4bc775,down,True
6,content_9a34b442b552,down,True
7,content_a63219c6e95a,stable,False
8,content_5e6c160719bc,down,True
9,content_c27558df2b0c,down,True


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule could work for something simple, like "flag every page where
impressions dropped by more than X%". But the real signals here are messy
and tangled together - things like position, CTR, content age, word count
and traffic all interact in ways that are hard to write as a clean
if-statement. This is exactly the kind of situation where ML earns its
place: the starter pipeline already proved this concretely - the plain
rule-based baseline only got 0.240 precision@50, while a random forest
model, trained on the same signals, hit 0.740 precision@50. That's a real,
measurable improvement, not just a theoretical one. So ML isn't just
"fancier", it's actually finding a pattern across multiple signals at once
that a single hand-written rule can't capture.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.